# GameTheory-16h : Coopération LLM — protocole multi-seed (Rust) (Vallinder–Hughes 2024, Payne–Alloui–Cros 2025)

**Jumeau Rust** du [carnet Python](GameTheory-16h-Cooperation-LLM-Python.ipynb) de la sous-série `GameTheory-16` (Mécanismes & agents LLM) — portage `std` (kernel `rust`/evcxr, aucun crate) : les 9 sorties numériques sont octet-identiques au jumeau Python. Ce carnet **étend** le pilote `GameTheory-16d` (Othman–Sandholm SAGT 2009, Mécanisme à paires) et la population multi-seed de `GameTheory-16e` au cas de la **coopération entre agents LLM** sur un **Dilemme du Prisonnier Itéré** (IPD, Axelrod 1984).

**Cadre** : un protocole reproductible qui **gèle les paramètres avant le premier appel d'agent**, mesure la coopération moyenne sur **≥ 4 hétérogénéités de population** (LLM-only / mix LLM+scripté / scripté pur / contrôle négatif), sur les seeds `{0, 1, 7, 42, 99}`, avec un **modèle nul** (`uniform_random`) et un **contrôle négatif** (`always_defect`).

**Sources** (gisement `G:\Mon Drive\MyIA\IA\Bibliographie IA\GameTheory\`) :

- Vallinder A., Hughes C. (2024) — *Cultural Evolution of Cooperation among LLM Agents*.
- Payne J., Alloui M., Cros A. (2025) — *Strategic Intelligence in Large Language Models*.

**Périmètre v1** : protocole + simulation CPU-only avec politiques scriptées qui **modélisent** les comportements LLM attendus (TFT — Tit-for-Tat, generous TFT, suspicious TFT, random). L'appel LLM réel est documenté §8 comme **extension future sur lane équipée** (Règle F : CPU-only, sans proxy LLM garanti sur cette lane).

**Acceptance** : 5 critères — protocole gelé avant tout, ≥ 2 hétérogénéités (4 retenues), multi-seed ≥ 4, modèle nul + contrôle négatif, verdicts H0/H1 par hétérogénéité.

**Mémo de gel** : [protocole 16h](../../docs/research/cooperation-llm-16h-protocole.md) — gel §0, hétérogénéités, seuils H0/H1, résultats mesurés, extension LLM.

**Navigation** : [Index GameTheory](README.md) | Précédent : [GameTheory-16e -- Populations Othman-Sandholm](GameTheory-16e-Populations-Othman-Sandholm-Python.ipynb) | Suivant : [GameTheory-16i -- Appariage stable (Gale-Shapley)](GameTheory-16i-Appariage-Stable-Gale-Shapley-Python.ipynb) | Pilote : [GameTheory-16d -- Joueurs LLM hétérogènes](GameTheory-16d-LLM-Players-Othman-Sandholm-Python.ipynb).


In [2]:
// =====================================================================
// §0 — gel du protocole (Vallinder–Hughes 2024, Payne–Alloui–Cros 2025)
// =====================================================================
const SEEDS: [i64; 5] = [0, 1, 7, 42, 99]; // sous-ensemble strict du jeu standard (au moins 4)
const N_ROUNDS: usize = 100; // longueur IPD (Axelrod 1984)
const N_MATCHES_PER_PAIR: usize = 50; // rencontres independantes par (heterogeneite, seed)

assert!(SEEDS.len() >= 4, "Multi-seed : au moins 4 seeds (jeu standard 0, 1, 7, 42, 99).");
assert!(N_ROUNDS >= 50, "IPD : N_ROUNDS >= 50 pour stabilite.");
assert!(N_MATCHES_PER_PAIR >= 30, "Echantillon : N_MATCHES_PER_PAIR >= 30 pour CLT.");

println!("Imports + gel preliminaire OK.");
println!("  SEEDS = {:?}", SEEDS);
println!("  N_ROUNDS = {}, N_MATCHES_PER_PAIR = {}", N_ROUNDS, N_MATCHES_PER_PAIR);

Imports + gel preliminaire OK.


  SEEDS = [0, 1, 7, 42, 99]


  N_ROUNDS = 100, N_MATCHES_PER_PAIR = 50


## Préambule — qu'est-ce qu'une *hétérogénéité de population* en IPD ?

Le pilote `GameTheory-16d` joue **une paire** (row, col) sur un mécanisme M ; le `GameTheory-16e` définit une **population** comme une distribution sur (θ, r) — un profil vrai et deux rapports d'action. Ici, on change de jeu : on joue un **Dilemme du Prisonnier Itéré** (IPD) et une *hétérogénéité* = une **distribution sur les stratégies** des agents qui peuplent la rencontre.

Trois hétérogénéités sont définies (§2) :

- **`llm_only`** : deux agents LLM, l'un TFT, l'autre Generous TFT. Modélisation CPU : un *proxy* de politique scriptée qui imite le comportement LLM dominant mesuré par Vallinder–Hughes 2024.
- **`mix_llm_scripted`** : un agent LLM (TFT) + un agent scripté (`tit_for_tat` strict). Modélise la **coopération inter-classes** (LLM + code déterministe).
- **`scripted_only`** : deux agents scriptés (TFT vs Generous TFT). Baseline de **coopération pure scriptée** (pas d'agent LLM dans la boucle).
- **`always_defect`** : deux agents `always_defect`. **Contrôle négatif** : doit dégrader la coopération à 0 %.

Le **modèle nul** `uniform_random_seed` (§5) sert de baseline inerte : chaque agent joue `C` ou `D` uniformément au hasard, indépendamment du passé — c'est la **strate inférieure** de la coopération.

**Pourquoi un proxy scripté ?** Le proxy reproduit le comportement LLM mesure par Vallinder-Hughes 2024 sans dependre d un appel API externe, ce qui rend le carnet reproductible sur CPU. Un audit peut substituer le LLM derrière le proxy (§8) sans réecrire la pipeline : le **noyau** de la valeur du carnet est le **protocole**, pas l agent LLM.

## 0. Pré-enregistrement du protocole (gelé avant tout appel d'agent)

Rôles, conditions, hétérogénéités, seeds, hypothèses, seuils de verdict et contrôles sont figés ici. Toute déviation exécution vs gel est consignée au §8 comme violation de protocole.

**Rôles** : `agent_0` (joueur 1) et `agent_1` (joueur 2) — symétrie d'information.

**Jeu** : Dilemme du Prisonnier Itéré (IPD), payoff `(T, R, P, S) = (5, 3, 1, 0)`. Axelrod 1984, tournoi informatique.

**Hétérogénéités (4, dont 1 contrôle négatif)** :

- `llm_only` | deux agents LLM : (TFT, GenerousTFT). Proxy scripté §1.
- `mix_llm_scripted` | un agent LLM (TFT) + un scripté (TFT strict). Hétérogénéité **inter-classes**.
- `scripted_only` | deux scriptés : (TFT, GenerousTFT). Baseline pure scriptée.
- `always_defect` | deux agents `always_defect`. **Contrôle négatif** — doit dégrader à 0 % de coopération.

**Politique d'information** : un agent observe l'historique complet des coups joués par l'adversaire (IPD canonique), pas d'information cachée.

**Seeds** : `{0, 1, 7, 42, 99}` — sous-ensemble strict de 4+ éléments du jeu standard (modèle #17845 héritage 16d/16e). 5 seeds pour la robustesse.

**Métriques** :

- `cooperation_rate(match_history)` : proportion de rounds où l'agent joue `C` (cooperate).
- `mean_coop_rate(pop, seed)` : moyenne sur les `N_MATCHES_PER_PAIR` rencontres indépendantes.
- `total_payoff` : somme des payoffs de l'agent sur les N_ROUNDS rounds (mesure secondaire, complémente la coopération).

**Seuils de verdict (H0/H1)** :

- **H0** — *la coopération moyenne est constante cross-hétérogénéité* — test : moyenne par seed dans ±0.05 du `uniform_random_seed` (≈ 0.5 attendu).
- **H1** — *la coopération moyenne dépend de l'hétérogénéité* — au moins une hétérogénéité sort de la bande ±0.05.

**Modèle nul** : `uniform_random_seed` — chaque round, l'agent joue `C` ou `D` uniformément au hasard (probabilité 0.5 chaque).

**Contrôle négatif** : `always_defect` — l'agent joue `D` à tous les rounds ; le **taux de coopération doit être exactement 0** (validation mécanique de la pipeline).

In [3]:
// §0 — gel du protocole : asserts d'auto-coherence (re-verification du gel)
const SEEDS: [i64; 5] = [0, 1, 7, 42, 99];
const N_ROUNDS: usize = 100;
const N_MATCHES_PER_PAIR: usize = 50;

assert!(SEEDS == [0, 1, 7, 42, 99], "Seeds gelees : 0, 1, 7, 42, 99.");
assert!(N_ROUNDS == 100, "N_ROUNDS gele : 100.");
assert!(N_MATCHES_PER_PAIR == 50, "N_MATCHES_PER_PAIR gele : 50.");

const HETEROGENEITIES: [&str; 4] = ["llm_only", "mix_llm_scripted", "scripted_only", "always_defect"];
assert!(HETEROGENEITIES.len() >= 2, "Au moins 2 heterogeneites (4 retenues).");
const NULL_MODEL: &str = "uniform_random_seed"; // baseline inerte (sert au §5)
const NEGATIVE_CONTROL: &str = "always_defect";
assert!(HETEROGENEITIES.contains(&NEGATIVE_CONTROL), "Le controle negatif doit figurer dans HETEROGENEITIES.");
const H0_NULL_BAND: f64 = 0.05; // moyenne dans +/- 0.05 du nul (0.5) => H0

fn py_list_s(v: &[&str]) -> String {
    let mut parts: Vec<String> = Vec::new();
    for s in v { parts.push(format!("'{}'", s)); }
    format!("[{}]", parts.join(", "))
}

println!("§0 protocole gele :");
println!("  SEEDS = {:?}", SEEDS);
println!("  N_ROUNDS = {}, N_MATCHES_PER_PAIR = {}", N_ROUNDS, N_MATCHES_PER_PAIR);
println!("  HETEROGENEITIES = {}", py_list_s(&HETEROGENEITIES));
println!("  NULL_MODEL = {} (modele nul, baseline §5)", NULL_MODEL);
println!("  NEGATIVE_CONTROL = {}", NEGATIVE_CONTROL);
println!("  H0 band = +/- {}", H0_NULL_BAND);

§0 protocole gele :


  SEEDS = [0, 1, 7, 42, 99]


  N_ROUNDS = 100, N_MATCHES_PER_PAIR = 50


  HETEROGENEITIES = ['llm_only', 'mix_llm_scripted', 'scripted_only', 'always_defect']


  NULL_MODEL = uniform_random_seed (modele nul, baseline §5)


  NEGATIVE_CONTROL = always_defect


  H0 band = +/- 0.05


## 1. Le jeu IPD et les politiques (proxy scripté des agents LLM)

Le **Dilemme du Prisonnier Itéré** est le substrat canonique des études de coopération (Axelrod 1984, *The Evolution of Cooperation*). Le payoff par round est `(T, R, P, S) = (5, 3, 1, 0)` — la condition `T > R > P > S` et `2R > T + S` garantit la **structure du dilemne** : la défection individuelle est tentante, mais la coopération mutuelle sur le long terme est collectivement meilleure.

**Politiques scriptées** (proxy des agents LLM, Vallinder–Hughes 2024) :

- **`tit_for_tat` (TFT)** : coopère au round 1, puis copie le dernier coup de l'adversaire. La politique la plus célèbre d'Axelrod 1984.
- **`generous_tft`** : TFT, mais avec probabilité `p_gen = 0.1` de **pardonner** une défection (jouer `C` quand l'adversaire a joué `D`). Modélise la **générosité** mesurée chez les LLM.
- **`suspicious_tft`** : défection au round 1, puis TFT. Variante testée par Axelrod 1984, moins performante que TFT.
- **`always_defect`** : joue `D` à tous les rounds. **Contrôle négatif** : doit dégrader à 0 % de coopération.
- **`uniform_random_seed`** : joue `C` ou `D` uniformément au hasard. **Modèle nul**.

**Hypothèse de modélisation** : un LLM *prompté* à coopérer joue en moyenne comme un `generous_tft` (cf Vallinder–Hughes 2024 §3.2 : les LLM montrent une **générosité** significativement supérieure à un TFT strict dans le tournoi IPD). Le proxy est volontairement **fruste** : l'**intérêt du protocole est dans la pipeline reproductible**, pas dans la finesse de la politique LLM (celle-ci relève du §8).

In [4]:
// §1 — Jeu IPD (T, R, P, S) et les 6 politiques, dont le proxy scripte des agents LLM
#[derive(Clone, Copy, PartialEq)]
enum Act { C, D }

const T: i64 = 5; // tentation
const R: i64 = 3; // recompense
const P: i64 = 1; // punition
const S: i64 = 0; // duperie

#[allow(dead_code)]
fn payoff(a0: Act, a1: Act) -> (i64, i64) {
    match (a0, a1) {
        (Act::C, Act::C) => (R, R),
        (Act::C, Act::D) => (S, T),
        (Act::D, Act::C) => (T, S),
        (Act::D, Act::D) => (P, P),
    }
}

// ===== A recopier tel quel dans chaque cellule qui tire au hasard (6, 12, 15, 18, 21, 24) =====
struct Mt19937 { mt: [u32; 624], idx: usize }

#[allow(dead_code)]
impl Mt19937 {
    fn init_genrand(s: u32) -> [u32; 624] {
        let mut mt = [0u32; 624];
        mt[0] = s;
        for i in 1..624 {
            mt[i] = 1812433253u32.wrapping_mul(mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_add(i as u32);
        }
        mt
    }
    fn new(seed: u32) -> Mt19937 { Self::new_by_array(&[seed]) }  // random.Random(seed), 0 <= seed < 2**32
    fn new_by_array(key: &[u32]) -> Mt19937 {     // init_by_array(key) de CPython
        let mut mt = Self::init_genrand(19650218);
        let (mut i, mut j) = (1usize, 0);
        for _ in 0..624.max(key.len()) {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1664525))
                .wrapping_add(key[j]).wrapping_add(j as u32);
            i += 1; j += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
            if j >= key.len() { j = 0; }
        }
        for _ in 0..623 {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1566083941))
                .wrapping_sub(i as u32);
            i += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
        }
        mt[0] = 0x8000_0000;
        Mt19937 { mt, idx: 624 }
    }
    fn next_u32(&mut self) -> u32 {
        if self.idx >= 624 {
            for k in 0..624 {
                let y = (self.mt[k] & 0x8000_0000) | (self.mt[(k + 1) % 624] & 0x7fff_ffff);
                let mut v = self.mt[(k + 397) % 624] ^ (y >> 1);
                if y & 1 != 0 { v ^= 0x9908_b0df; }
                self.mt[k] = v;
            }
            self.idx = 0;
        }
        let mut y = self.mt[self.idx];
        self.idx += 1;
        y ^= y >> 11;
        y ^= (y << 7) & 0x9d2c_5680;
        y ^= (y << 15) & 0xefc6_0000;
        y ^= y >> 18;
        y
    }
    fn random(&mut self) -> f64 {              // random.random() : 53 bits
        let a = (self.next_u32() >> 5) as f64;
        let b = (self.next_u32() >> 6) as f64;
        (a * 67108864.0 + b) / 9007199254740992.0
    }
    fn getrandbits(&mut self, k: u32) -> u64 {  // 1 <= k <= 64
        if k <= 32 { return (self.next_u32() >> (32 - k)) as u64; }
        let lo = self.next_u32() as u64;
        let hi = (self.next_u32() >> (64 - k)) as u64;
        lo | (hi << 32)
    }
    fn randbelow(&mut self, n: u64) -> u64 {     // rejet sur getrandbits(n.bit_length())
        let k = 64 - n.leading_zeros();
        let mut r = self.getrandbits(k);
        while r >= n { r = self.getrandbits(k); }
        r
    }
    fn randint(&mut self, a: i64, b: i64) -> i64 { a + self.randbelow((b - a + 1) as u64) as i64 }
    fn choice<'a, T>(&mut self, seq: &'a [T]) -> &'a T { &seq[self.randbelow(seq.len() as u64) as usize] }
    fn shuffle<T>(&mut self, x: &mut [T]) {
        for i in (1..x.len()).rev() { let j = self.randbelow(i as u64 + 1) as usize; x.swap(i, j); }
    }
}

#[allow(dead_code)]
fn policy(name: &str, hist: &[(Act, Act)], rng: &mut Mt19937, role: usize) -> Act {
    // dernier coup de l'autre agent (role 0 lit le coup de l'agent 1, et reciproque)
    let other_last = |h: &[(Act, Act)], r: usize| -> Act {
        let last = &h[h.len() - 1];
        if r == 0 { last.1 } else { last.0 }
    };
    match name {
        "tft" => {
            if hist.is_empty() { Act::C } else { other_last(hist, role) }
        }
        "generous_tft" | "llm_proxy_tft" => { // llm_proxy_tft delegue a generous_tft (proxy Vallinder-Hughes 2024)
            if hist.is_empty() {
                Act::C
            } else {
                let o = other_last(hist, role);
                if o == Act::D && rng.random() < 0.1 { Act::C } else { o }
            }
        }
        "suspicious_tft" => {
            if hist.is_empty() { Act::D } else { other_last(hist, role) }
        }
        "always_defect" => Act::D,
        "uniform_random_seed" => *rng.choice(&[Act::C, Act::D]),
        _ => Act::D, // nom inconnu : jamais appele ici (les 6 noms ci-dessus couvrent le carnet)
    }
}

fn py_list_s(v: &[&str]) -> String {
    let mut parts: Vec<String> = Vec::new();
    for s in v { parts.push(format!("'{}'", s)); }
    format!("[{}]", parts.join(", "))
}

const POLICY_NAMES: [&str; 6] = ["tft", "generous_tft", "suspicious_tft", "always_defect", "uniform_random_seed", "llm_proxy_tft"];
assert!(POLICY_NAMES.len() >= 5, "Au moins 5 politiques chargees.");

println!("§1 — {} politiques chargees : {}", POLICY_NAMES.len(), py_list_s(&POLICY_NAMES));
println!("  Payoff matrix : T={}, R={}, P={}, S={}", T, R, P, S);
println!("  Structure dilemme : T > R > P > S, 2R > T + S ({} > {})", 2 * R, T + S);

§1 — 6 politiques chargees : ['tft', 'generous_tft', 'suspicious_tft', 'always_defect', 'uniform_random_seed', 'llm_proxy_tft']


  Payoff matrix : T=5, R=3, P=1, S=0


  Structure dilemme : T > R > P > S, 2R > T + S (6 > 5)


**Lecture — §1.** Cinq politiques sont définies : trois variantes TFT (strict, généreux, suspect), un `always_defect` (contrôle négatif), et un `uniform_random_seed` (modèle nul). Le **proxy LLM** est `generous_tft` (Vallinder–Hughes 2024 ont mesuré que les LLM *promptés* à coopérer ont une **générosité** significativement supérieure à un TFT strict dans le tournoi IPD).

La **structure du dilemne** est satisfaite par les valeurs choisies :

- `T > R` (5 > 3) : la défection contre un coopérant rapporte plus que la coopération mutuelle.
- `R > P` (3 > 1) : la coopération mutuelle est meilleure que la défection mutuelle.
- `P > S` (1 > 0) : la défection unilatérale est moins punie que la coopération unilatérale.
- `2R > T + S` (6 > 5) : la coopération mutuelle Itérée bat l'alternance exploitation/exploitation.

## 2. Définition d'une hétérogénéité — distribution sur (strat_0, strat_1)

Une **hétérogénéité** = un **couple de politiques** `(strat_0, strat_1)` qui déterminent les deux agents de la rencontre IPD. Le gel §0 a listé les 4 hétérogénéités canoniques ; ce §2 les implémente.

**Conventions** :

- `llm_only` = `(llm_proxy_tft, generous_tft)` : deux "agents LLM" (proxy généreux).
- `mix_llm_scripted` = `(llm_proxy_tft, tit_for_tat)` : un LLM + un scripté strict.
- `scripted_only` = `(generous_tft, suspicious_tft)` : deux scriptés, hétérogènes (générosité vs suspicion).
- `always_defect` = `(always_defect, always_defect)` : contrôle négatif — doit dégrader à 0 %.

In [5]:
// §2 — Heterogeneites : couples de politiques nommees (cle = nom de politique du §1)
const HETEROGENEITIES: [&str; 4] = ["llm_only", "mix_llm_scripted", "scripted_only", "always_defect"];
// Sequence ORDONNEE (jamais trie : le tableau du §4 suit cet ordre, pas l'alpha)
const HET_PAIRS: [&(&str, &str, &str); 4] = [
    &("llm_only", "llm_proxy_tft", "generous_tft"),
    &("mix_llm_scripted", "llm_proxy_tft", "tft"),
    &("scripted_only", "generous_tft", "suspicious_tft"),
    &("always_defect", "always_defect", "always_defect"),
];

for &(het, _, _) in HET_PAIRS {
    assert!(HETEROGENEITIES.contains(&het), "HET_PAIRS doit couvrir HETEROGENEITIES.");
};

println!("§2 — Heterogeneites :");
for &(het, s0, s1) in HET_PAIRS {
    println!("  {:20} = ({}, {})", het, s0, s1);
};

§2 — Heterogeneites :


  llm_only             = (llm_proxy_tft, generous_tft)


  mix_llm_scripted     = (llm_proxy_tft, tft)


  scripted_only        = (generous_tft, suspicious_tft)


  always_defect        = (always_defect, always_defect)


**Lecture — §2.** Les 4 hétérogénéités sont définies par un couple de politiques du registre `POLICIES`. **Vérification de cohérence** : le `assert set(HET_PAIRS.keys()) == set(HETEROGENEITIES)` garantit qu'aucune hétérogénéité du gel §0 n'est oubliée à l'implémentation.

**`mix_llm_scripted`** est l'hétérogénéité la plus intéressante du point de vue de la **sélection culturelle** (Vallinder–Hughes 2024) : un agent LLM (généreux) face à un TFT strict. Le TFT strict **n pardonne pas**, donc la moindre défection du LLM déclenche un cycle de représailles. La **générosité** du LLM est ici **pénalisée** par le scripté strict — l'inverse de la dynamique `llm_only` où les deux LLM se pardonnent mutuellement.

## 3. Driver multi-seed — exécution reproductible d'une rencontre IPD

Pour chaque `(heterogeneite, seed)`, on tire `N_MATCHES_PER_PAIR` rencontres **indépendantes** de `N_ROUNDS` rounds chacune. À chaque round, chaque agent choisit son coup selon sa politique et l'historique. On accumule le **taux de coopération** de chaque agent (moyenne sur les N_ROUNDS rounds) et le **payoff total**.

**Hypothèse** : le RNG est un **MT19937 bit-exact** de `random.Random(seed)` de CPython (bloc `Mt19937` recopié tel quel dans chaque cellule qui tire) — déterminisme garanti par seed, quel que soit l'état global.

In [6]:
// §3 — Driver multi-seed : rencontres IPD independantes par (heterogeneite, seed)
#[allow(dead_code)]
const SEEDS: [i64; 5] = [0, 1, 7, 42, 99];
const N_ROUNDS: usize = 100;
#[allow(dead_code)]
const N_MATCHES_PER_PAIR: usize = 50;
const T: i64 = 5;
const R: i64 = 3;
const P: i64 = 1;
const S: i64 = 0;

#[derive(Clone, Copy, PartialEq)]
enum Act { C, D }

fn payoff(a0: Act, a1: Act) -> (i64, i64) {
    match (a0, a1) {
        (Act::C, Act::C) => (R, R),
        (Act::C, Act::D) => (S, T),
        (Act::D, Act::C) => (T, S),
        (Act::D, Act::D) => (P, P),
    }
}

// ===== A recopier tel quel dans chaque cellule qui tire au hasard (6, 12, 15, 18, 21, 24) =====
struct Mt19937 { mt: [u32; 624], idx: usize }

#[allow(dead_code)]
impl Mt19937 {
    fn init_genrand(s: u32) -> [u32; 624] {
        let mut mt = [0u32; 624];
        mt[0] = s;
        for i in 1..624 {
            mt[i] = 1812433253u32.wrapping_mul(mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_add(i as u32);
        }
        mt
    }
    fn new(seed: u32) -> Mt19937 { Self::new_by_array(&[seed]) }  // random.Random(seed), 0 <= seed < 2**32
    fn new_by_array(key: &[u32]) -> Mt19937 {     // init_by_array(key) de CPython
        let mut mt = Self::init_genrand(19650218);
        let (mut i, mut j) = (1usize, 0);
        for _ in 0..624.max(key.len()) {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1664525))
                .wrapping_add(key[j]).wrapping_add(j as u32);
            i += 1; j += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
            if j >= key.len() { j = 0; }
        }
        for _ in 0..623 {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1566083941))
                .wrapping_sub(i as u32);
            i += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
        }
        mt[0] = 0x8000_0000;
        Mt19937 { mt, idx: 624 }
    }
    fn next_u32(&mut self) -> u32 {
        if self.idx >= 624 {
            for k in 0..624 {
                let y = (self.mt[k] & 0x8000_0000) | (self.mt[(k + 1) % 624] & 0x7fff_ffff);
                let mut v = self.mt[(k + 397) % 624] ^ (y >> 1);
                if y & 1 != 0 { v ^= 0x9908_b0df; }
                self.mt[k] = v;
            }
            self.idx = 0;
        }
        let mut y = self.mt[self.idx];
        self.idx += 1;
        y ^= y >> 11;
        y ^= (y << 7) & 0x9d2c_5680;
        y ^= (y << 15) & 0xefc6_0000;
        y ^= y >> 18;
        y
    }
    fn random(&mut self) -> f64 {              // random.random() : 53 bits
        let a = (self.next_u32() >> 5) as f64;
        let b = (self.next_u32() >> 6) as f64;
        (a * 67108864.0 + b) / 9007199254740992.0
    }
    fn getrandbits(&mut self, k: u32) -> u64 {  // 1 <= k <= 64
        if k <= 32 { return (self.next_u32() >> (32 - k)) as u64; }
        let lo = self.next_u32() as u64;
        let hi = (self.next_u32() >> (64 - k)) as u64;
        lo | (hi << 32)
    }
    fn randbelow(&mut self, n: u64) -> u64 {     // rejet sur getrandbits(n.bit_length())
        let k = 64 - n.leading_zeros();
        let mut r = self.getrandbits(k);
        while r >= n { r = self.getrandbits(k); }
        r
    }
    fn randint(&mut self, a: i64, b: i64) -> i64 { a + self.randbelow((b - a + 1) as u64) as i64 }
    fn choice<'a, T>(&mut self, seq: &'a [T]) -> &'a T { &seq[self.randbelow(seq.len() as u64) as usize] }
    fn shuffle<T>(&mut self, x: &mut [T]) {
        for i in (1..x.len()).rev() { let j = self.randbelow(i as u64 + 1) as usize; x.swap(i, j); }
    }
}

fn policy(name: &str, hist: &[(Act, Act)], rng: &mut Mt19937, role: usize) -> Act {
    // dernier coup de l'autre agent (role 0 lit le coup de l'agent 1, et reciproque)
    let other_last = |h: &[(Act, Act)], r: usize| -> Act {
        let last = &h[h.len() - 1];
        if r == 0 { last.1 } else { last.0 }
    };
    match name {
        "tft" => {
            if hist.is_empty() { Act::C } else { other_last(hist, role) }
        }
        "generous_tft" | "llm_proxy_tft" => { // llm_proxy_tft delegue a generous_tft (proxy Vallinder-Hughes 2024)
            if hist.is_empty() {
                Act::C
            } else {
                let o = other_last(hist, role);
                if o == Act::D && rng.random() < 0.1 { Act::C } else { o }
            }
        }
        "suspicious_tft" => {
            if hist.is_empty() { Act::D } else { other_last(hist, role) }
        }
        "always_defect" => Act::D,
        "uniform_random_seed" => *rng.choice(&[Act::C, Act::D]),
        _ => Act::D, // nom inconnu : jamais appele ici (les 6 noms ci-dessus couvrent le carnet)
    }
}

fn py_f(x: f64) -> String { format!("{:?}", x) } // 0.0 -> "0.0" (identique au repr Python)

#[allow(dead_code)]
const HETEROGENEITIES: [&str; 4] = ["llm_only", "mix_llm_scripted", "scripted_only", "always_defect"];
// Sequence ORDONNEE (jamais trie : le tableau du §4 suit cet ordre, pas l'alpha)
const HET_PAIRS: [&(&str, &str, &str); 4] = [
    &("llm_only", "llm_proxy_tft", "generous_tft"),
    &("mix_llm_scripted", "llm_proxy_tft", "tft"),
    &("scripted_only", "generous_tft", "suspicious_tft"),
    &("always_defect", "always_defect", "always_defect"),
];

struct MatchRow {
    coop_rate_0: f64,
    coop_rate_1: f64,
    payoff_0: i64,
    payoff_1: i64,
}

fn run_match(strat_0: &str, strat_1: &str, seed: i64) -> MatchRow {
    let mut rng = Mt19937::new(seed as u32);
    let mut history: Vec<(Act, Act)> = Vec::new();
    let (mut pay_0, mut pay_1) = (0i64, 0i64);
    let (mut coop_0, mut coop_1) = (0usize, 0usize);
    for _ in 0..N_ROUNDS {
        let a0 = policy(strat_0, &history, &mut rng, 0);
        let a1 = policy(strat_1, &history, &mut rng, 1);
        history.push((a0, a1));
        let (u0, u1) = payoff(a0, a1);
        pay_0 += u0;
        pay_1 += u1;
        if a0 == Act::C { coop_0 += 1; }
        if a1 == Act::C { coop_1 += 1; }
    }
    MatchRow {
        coop_rate_0: coop_0 as f64 / N_ROUNDS as f64,
        coop_rate_1: coop_1 as f64 / N_ROUNDS as f64,
        payoff_0: pay_0,
        payoff_1: pay_1,
    }
}

fn run_heterogeneity(het: &str, seed: i64, n_matches: usize) -> Vec<MatchRow> {
    let pair = HET_PAIRS
        .iter()
        .find(|t| t.0 == het)
        .copied()
        .unwrap_or(&("always_defect", "always_defect", "always_defect"));
    let (s0, s1) = (pair.1, pair.2);
    let mut rows: Vec<MatchRow> = Vec::new();
    for m in 0..n_matches {
        rows.push(run_match(s0, s1, seed * 1000 + m as i64));
    }
    rows
}

// Test negatif : 1 match always_defect, seed 0 (canari de la pipeline)
let test_rows = run_heterogeneity("always_defect", 0, 1);
assert!(test_rows[0].coop_rate_0 == 0.0 && test_rows[0].coop_rate_1 == 0.0, "Canari : always_defect doit cooperer a 0 %.");
println!("§3 — Driver OK. Test controle negatif :");
println!("  always_defect seed=0 : coop_0 = {}, coop_1 = {}", py_f(test_rows[0].coop_rate_0), py_f(test_rows[0].coop_rate_1));
println!("  (attendu : 0.0 / 0.0)");
println!("  payoff_0 = {}, payoff_1 = {}", test_rows[0].payoff_0, test_rows[0].payoff_1);
println!("  (attendu : {} / {} = defection mutuelle)", P * N_ROUNDS as i64, P * N_ROUNDS as i64);

§3 — Driver OK. Test controle negatif :


  always_defect seed=0 : coop_0 = 0.0, coop_1 = 0.0


  (attendu : 0.0 / 0.0)


  payoff_0 = 100, payoff_1 = 100


  (attendu : 100 / 100 = defection mutuelle)


**Lecture — §3.** Le driver est paramétré par `(heterogeneite, seed)` et exécute `N_MATCHES_PER_PAIR = 50` rencontres indépendantes. Le **combinateur de seed** `seed * 1000 + m` évite les collisions entre matches d'une même hétérogénéité.

**Test du contrôle négatif** : `always_defect` doit produire un taux de coopération de **0 %** et un payoff de `P * N_ROUNDS = 1 * 100 = 100` (défection mutuelle à chaque round). C'est le **canari de la pipeline** : si cette assertion échoue, tout le reste est invalide.

**Coût de calcul** : 4 hétérogénéités × 5 seeds × 50 matches × 100 rounds = **100 000 rounds** IPD au total. Très léger pour un notebook (≪ 1 s, calcul natif).

## 4. Agrégation par (hétérogénéité, seed) — distribution du taux de coopération

Pour chaque `(heterogeneite, seed)`, on agrège les `N_MATCHES_PER_PAIR` résultats en **moyenne et écart-type** du taux de coopération de chaque agent. L'output est un tableau `(het, seed) -> (mean_coop_0, std_coop_0, mean_coop_1, std_coop_1, mean_payoff_0, mean_payoff_1)`.

In [7]:
// §4 — Aggregation par (heterogeneite, seed) : 50 matches re-joues par (het, seed)
const SEEDS: [i64; 5] = [0, 1, 7, 42, 99];
const N_ROUNDS: usize = 100;
const N_MATCHES_PER_PAIR: usize = 50;
const T: i64 = 5;
const R: i64 = 3;
const P: i64 = 1;
const S: i64 = 0;

#[derive(Clone, Copy, PartialEq)]
enum Act { C, D }

fn payoff(a0: Act, a1: Act) -> (i64, i64) {
    match (a0, a1) {
        (Act::C, Act::C) => (R, R),
        (Act::C, Act::D) => (S, T),
        (Act::D, Act::C) => (T, S),
        (Act::D, Act::D) => (P, P),
    }
}

// ===== A recopier tel quel dans chaque cellule qui tire au hasard (6, 12, 15, 18, 21, 24) =====
struct Mt19937 { mt: [u32; 624], idx: usize }

#[allow(dead_code)]
impl Mt19937 {
    fn init_genrand(s: u32) -> [u32; 624] {
        let mut mt = [0u32; 624];
        mt[0] = s;
        for i in 1..624 {
            mt[i] = 1812433253u32.wrapping_mul(mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_add(i as u32);
        }
        mt
    }
    fn new(seed: u32) -> Mt19937 { Self::new_by_array(&[seed]) }  // random.Random(seed), 0 <= seed < 2**32
    fn new_by_array(key: &[u32]) -> Mt19937 {     // init_by_array(key) de CPython
        let mut mt = Self::init_genrand(19650218);
        let (mut i, mut j) = (1usize, 0);
        for _ in 0..624.max(key.len()) {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1664525))
                .wrapping_add(key[j]).wrapping_add(j as u32);
            i += 1; j += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
            if j >= key.len() { j = 0; }
        }
        for _ in 0..623 {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1566083941))
                .wrapping_sub(i as u32);
            i += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
        }
        mt[0] = 0x8000_0000;
        Mt19937 { mt, idx: 624 }
    }
    fn next_u32(&mut self) -> u32 {
        if self.idx >= 624 {
            for k in 0..624 {
                let y = (self.mt[k] & 0x8000_0000) | (self.mt[(k + 1) % 624] & 0x7fff_ffff);
                let mut v = self.mt[(k + 397) % 624] ^ (y >> 1);
                if y & 1 != 0 { v ^= 0x9908_b0df; }
                self.mt[k] = v;
            }
            self.idx = 0;
        }
        let mut y = self.mt[self.idx];
        self.idx += 1;
        y ^= y >> 11;
        y ^= (y << 7) & 0x9d2c_5680;
        y ^= (y << 15) & 0xefc6_0000;
        y ^= y >> 18;
        y
    }
    fn random(&mut self) -> f64 {              // random.random() : 53 bits
        let a = (self.next_u32() >> 5) as f64;
        let b = (self.next_u32() >> 6) as f64;
        (a * 67108864.0 + b) / 9007199254740992.0
    }
    fn getrandbits(&mut self, k: u32) -> u64 {  // 1 <= k <= 64
        if k <= 32 { return (self.next_u32() >> (32 - k)) as u64; }
        let lo = self.next_u32() as u64;
        let hi = (self.next_u32() >> (64 - k)) as u64;
        lo | (hi << 32)
    }
    fn randbelow(&mut self, n: u64) -> u64 {     // rejet sur getrandbits(n.bit_length())
        let k = 64 - n.leading_zeros();
        let mut r = self.getrandbits(k);
        while r >= n { r = self.getrandbits(k); }
        r
    }
    fn randint(&mut self, a: i64, b: i64) -> i64 { a + self.randbelow((b - a + 1) as u64) as i64 }
    fn choice<'a, T>(&mut self, seq: &'a [T]) -> &'a T { &seq[self.randbelow(seq.len() as u64) as usize] }
    fn shuffle<T>(&mut self, x: &mut [T]) {
        for i in (1..x.len()).rev() { let j = self.randbelow(i as u64 + 1) as usize; x.swap(i, j); }
    }
}

fn policy(name: &str, hist: &[(Act, Act)], rng: &mut Mt19937, role: usize) -> Act {
    // dernier coup de l'autre agent (role 0 lit le coup de l'agent 1, et reciproque)
    let other_last = |h: &[(Act, Act)], r: usize| -> Act {
        let last = &h[h.len() - 1];
        if r == 0 { last.1 } else { last.0 }
    };
    match name {
        "tft" => {
            if hist.is_empty() { Act::C } else { other_last(hist, role) }
        }
        "generous_tft" | "llm_proxy_tft" => { // llm_proxy_tft delegue a generous_tft (proxy Vallinder-Hughes 2024)
            if hist.is_empty() {
                Act::C
            } else {
                let o = other_last(hist, role);
                if o == Act::D && rng.random() < 0.1 { Act::C } else { o }
            }
        }
        "suspicious_tft" => {
            if hist.is_empty() { Act::D } else { other_last(hist, role) }
        }
        "always_defect" => Act::D,
        "uniform_random_seed" => *rng.choice(&[Act::C, Act::D]),
        _ => Act::D, // nom inconnu : jamais appele ici (les 6 noms ci-dessus couvrent le carnet)
    }
}

fn mean_f(v: &[f64]) -> f64 { let mut s = 0.0f64; for &x in v { s += x; } s / v.len() as f64 }
fn stdev_f(v: &[f64]) -> f64 {
    let m = mean_f(v);
    let mut s = 0.0f64;
    for &x in v { s += (x - m) * (x - m); }
    (s / (v.len() - 1) as f64).sqrt()
}

const HETEROGENEITIES: [&str; 4] = ["llm_only", "mix_llm_scripted", "scripted_only", "always_defect"];
// Sequence ORDONNEE (jamais trie : le tableau du §4 suit cet ordre, pas l'alpha)
const HET_PAIRS: [&(&str, &str, &str); 4] = [
    &("llm_only", "llm_proxy_tft", "generous_tft"),
    &("mix_llm_scripted", "llm_proxy_tft", "tft"),
    &("scripted_only", "generous_tft", "suspicious_tft"),
    &("always_defect", "always_defect", "always_defect"),
];

struct MatchRow {
    coop_rate_0: f64,
    coop_rate_1: f64,
    payoff_0: i64,
    payoff_1: i64,
}

fn run_match(strat_0: &str, strat_1: &str, seed: i64) -> MatchRow {
    let mut rng = Mt19937::new(seed as u32);
    let mut history: Vec<(Act, Act)> = Vec::new();
    let (mut pay_0, mut pay_1) = (0i64, 0i64);
    let (mut coop_0, mut coop_1) = (0usize, 0usize);
    for _ in 0..N_ROUNDS {
        let a0 = policy(strat_0, &history, &mut rng, 0);
        let a1 = policy(strat_1, &history, &mut rng, 1);
        history.push((a0, a1));
        let (u0, u1) = payoff(a0, a1);
        pay_0 += u0;
        pay_1 += u1;
        if a0 == Act::C { coop_0 += 1; }
        if a1 == Act::C { coop_1 += 1; }
    }
    MatchRow {
        coop_rate_0: coop_0 as f64 / N_ROUNDS as f64,
        coop_rate_1: coop_1 as f64 / N_ROUNDS as f64,
        payoff_0: pay_0,
        payoff_1: pay_1,
    }
}

fn run_heterogeneity(het: &str, seed: i64, n_matches: usize) -> Vec<MatchRow> {
    let pair = HET_PAIRS
        .iter()
        .find(|t| t.0 == het)
        .copied()
        .unwrap_or(&("always_defect", "always_defect", "always_defect"));
    let (s0, s1) = (pair.1, pair.2);
    let mut rows: Vec<MatchRow> = Vec::new();
    for m in 0..n_matches {
        rows.push(run_match(s0, s1, seed * 1000 + m as i64));
    }
    rows
}

// Balayage ordonne : 4 heterogeneites x 5 seeds
let mut rows_all: Vec<(&str, i64, f64, f64, f64, f64, f64)> = Vec::new();
for het in HETEROGENEITIES {
    for &seed in &SEEDS {
        let rows = run_heterogeneity(het, seed, N_MATCHES_PER_PAIR);
        let coop_0: Vec<f64> = rows.iter().map(|r| r.coop_rate_0).collect();
        let coop_1: Vec<f64> = rows.iter().map(|r| r.coop_rate_1).collect();
        let payoff_0: Vec<f64> = rows.iter().map(|r| r.payoff_0 as f64).collect();
        let _payoff_1: Vec<f64> = rows.iter().map(|r| r.payoff_1 as f64).collect();
        rows_all.push((het, seed, mean_f(&coop_0), stdev_f(&coop_0), mean_f(&coop_1), stdev_f(&coop_1), mean_f(&payoff_0)));
    }
};

println!("§4 — Resultats agreges par (heterogeneite, seed) :");
println!("  {:20} {:>5} {:>10} {:>8} {:>10} {:>8} {:>10}", "het", "seed", "mean_c0", "std_c0", "mean_c1", "std_c1", "payoff0");
for &(het, seed, mc0, sc0, mc1, sc1, mp0) in &rows_all {
    println!("  {:20} {:>5} {:>10.4} {:>8.4} {:>10.4} {:>8.4} {:>10.2}", het, seed, mc0, sc0, mc1, sc1, mp0);
};

§4 — Resultats agreges par (heterogeneite, seed) :


  het                   seed    mean_c0   std_c0    mean_c1   std_c1    payoff0


  llm_only                 0     1.0000   0.0000     1.0000   0.0000     300.00


  llm_only                 1     1.0000   0.0000     1.0000   0.0000     300.00


  llm_only                 7     1.0000   0.0000     1.0000   0.0000     300.00


  llm_only                42     1.0000   0.0000     1.0000   0.0000     300.00


... (12 outputs left out)

  always_defect            1     0.0000   0.0000     0.0000   0.0000     100.00


  always_defect            7     0.0000   0.0000     0.0000   0.0000     100.00


  always_defect           42     0.0000   0.0000     0.0000   0.0000     100.00


  always_defect           99     0.0000   0.0000     0.0000   0.0000     100.00


**Lecture — §4.** Le tableau croisé `(heterogeneite, seed)` montre trois patterns clés, calibrés sur les outputs réels :



1. **`llm_only`** : taux de coopération exactement **1.0000** sur les 5 seeds (`std_coop_0 = std_coop_1 = 0.0000`) — les deux LLM proxy (généreux TFT) se pardonnent mutuellement dès le round 1 et n'ont jamais l'occasion de dévier (la généreuse TFT ne dévie que si l'adversaire a déjà dévié deux fois, ce qui n'arrive jamais face à un généreux TFT).

2. **`mix_llm_scripted`** : taux de coopération exactement **1.0000** sur les 5 seeds (`std_coop_0 = std_coop_1 = 0.0000`) — le LLM proxy et le TFT strict convergent en coopération mutuelle dès le round 1 sans jamais dévier. **Le mécanisme « TFT strict peut dégrader la coopération si le LLM fait une défection » ne se déclenche pas en pratique** : le LLM proxy est généreux (1.0000, jamais de défection), donc le TFT strict n'a aucune raison de dévier non plus. 1.0000 est un point fixe trivial, pas une dynamique de pardon observée.

3. **`scripted_only`** (GenerousTFT vs SuspiciousTFT) : taux **0.9187 ± 0.0104** (moyenne par paire des deux agents, par seed), range intra-seed `mean_c0` **0.9114-0.9376** sur les 5 seeds (agent 0 seul). Le Suspicious commence par D, GenerousTFT copie (TFT) → défection mutuelle initiale au round 1, puis Generous pardonne (10% de chance de C après un D adverse) → convergence lente vers coopération, mais avec une traînée résiduelle de défection (~0.08) héritée du round 1.

4. **`always_defect`** : 0.0 / 0.0 sur tous les seeds (contrôle négatif). **Validation pipeline OK.**



**Convergence inter-seed** : l'écart-type intra-seed est très faible (max 0.09 sur `scripted_only`, 0 sur `llm_only`/`mix_llm_scripted`/`always_defect`) grâce à `N_MATCHES_PER_PAIR = 50` — la moyenne par seed est stable. La **vraie variabilité** est inter-hétérogénéité, pas inter-seed : c'est la conclusion attendue du gel §0.

## 5. Modèle nul — `uniform_random_seed`

Le **modèle nul** `uniform_random_seed` (cf §1) est la baseline inerte : chaque round, l'agent joue `C` ou `D` avec probabilité 0.5. Le **taux de coopération attendu** est exactement **0.5** par construction. C'est la **strate inférieure** contre laquelle les hétérogénéités sont comparées au §7.

In [8]:
// §5 — Modele nul : uniform_random_seed vs uniform_random_seed (baseline inerte)
const SEEDS: [i64; 5] = [0, 1, 7, 42, 99];
const N_ROUNDS: usize = 100;
const N_MATCHES_PER_PAIR: usize = 50;
const T: i64 = 5;
const R: i64 = 3;
const P: i64 = 1;
const S: i64 = 0;

#[derive(Clone, Copy, PartialEq)]
enum Act { C, D }

fn payoff(a0: Act, a1: Act) -> (i64, i64) {
    match (a0, a1) {
        (Act::C, Act::C) => (R, R),
        (Act::C, Act::D) => (S, T),
        (Act::D, Act::C) => (T, S),
        (Act::D, Act::D) => (P, P),
    }
}

// ===== A recopier tel quel dans chaque cellule qui tire au hasard (6, 12, 15, 18, 21, 24) =====
struct Mt19937 { mt: [u32; 624], idx: usize }

#[allow(dead_code)]
impl Mt19937 {
    fn init_genrand(s: u32) -> [u32; 624] {
        let mut mt = [0u32; 624];
        mt[0] = s;
        for i in 1..624 {
            mt[i] = 1812433253u32.wrapping_mul(mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_add(i as u32);
        }
        mt
    }
    fn new(seed: u32) -> Mt19937 { Self::new_by_array(&[seed]) }  // random.Random(seed), 0 <= seed < 2**32
    fn new_by_array(key: &[u32]) -> Mt19937 {     // init_by_array(key) de CPython
        let mut mt = Self::init_genrand(19650218);
        let (mut i, mut j) = (1usize, 0);
        for _ in 0..624.max(key.len()) {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1664525))
                .wrapping_add(key[j]).wrapping_add(j as u32);
            i += 1; j += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
            if j >= key.len() { j = 0; }
        }
        for _ in 0..623 {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1566083941))
                .wrapping_sub(i as u32);
            i += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
        }
        mt[0] = 0x8000_0000;
        Mt19937 { mt, idx: 624 }
    }
    fn next_u32(&mut self) -> u32 {
        if self.idx >= 624 {
            for k in 0..624 {
                let y = (self.mt[k] & 0x8000_0000) | (self.mt[(k + 1) % 624] & 0x7fff_ffff);
                let mut v = self.mt[(k + 397) % 624] ^ (y >> 1);
                if y & 1 != 0 { v ^= 0x9908_b0df; }
                self.mt[k] = v;
            }
            self.idx = 0;
        }
        let mut y = self.mt[self.idx];
        self.idx += 1;
        y ^= y >> 11;
        y ^= (y << 7) & 0x9d2c_5680;
        y ^= (y << 15) & 0xefc6_0000;
        y ^= y >> 18;
        y
    }
    fn random(&mut self) -> f64 {              // random.random() : 53 bits
        let a = (self.next_u32() >> 5) as f64;
        let b = (self.next_u32() >> 6) as f64;
        (a * 67108864.0 + b) / 9007199254740992.0
    }
    fn getrandbits(&mut self, k: u32) -> u64 {  // 1 <= k <= 64
        if k <= 32 { return (self.next_u32() >> (32 - k)) as u64; }
        let lo = self.next_u32() as u64;
        let hi = (self.next_u32() >> (64 - k)) as u64;
        lo | (hi << 32)
    }
    fn randbelow(&mut self, n: u64) -> u64 {     // rejet sur getrandbits(n.bit_length())
        let k = 64 - n.leading_zeros();
        let mut r = self.getrandbits(k);
        while r >= n { r = self.getrandbits(k); }
        r
    }
    fn randint(&mut self, a: i64, b: i64) -> i64 { a + self.randbelow((b - a + 1) as u64) as i64 }
    fn choice<'a, T>(&mut self, seq: &'a [T]) -> &'a T { &seq[self.randbelow(seq.len() as u64) as usize] }
    fn shuffle<T>(&mut self, x: &mut [T]) {
        for i in (1..x.len()).rev() { let j = self.randbelow(i as u64 + 1) as usize; x.swap(i, j); }
    }
}

fn policy(name: &str, hist: &[(Act, Act)], rng: &mut Mt19937, role: usize) -> Act {
    // dernier coup de l'autre agent (role 0 lit le coup de l'agent 1, et reciproque)
    let other_last = |h: &[(Act, Act)], r: usize| -> Act {
        let last = &h[h.len() - 1];
        if r == 0 { last.1 } else { last.0 }
    };
    match name {
        "tft" => {
            if hist.is_empty() { Act::C } else { other_last(hist, role) }
        }
        "generous_tft" | "llm_proxy_tft" => { // llm_proxy_tft delegue a generous_tft (proxy Vallinder-Hughes 2024)
            if hist.is_empty() {
                Act::C
            } else {
                let o = other_last(hist, role);
                if o == Act::D && rng.random() < 0.1 { Act::C } else { o }
            }
        }
        "suspicious_tft" => {
            if hist.is_empty() { Act::D } else { other_last(hist, role) }
        }
        "always_defect" => Act::D,
        "uniform_random_seed" => *rng.choice(&[Act::C, Act::D]),
        _ => Act::D, // nom inconnu : jamais appele ici (les 6 noms ci-dessus couvrent le carnet)
    }
}

fn mean_f(v: &[f64]) -> f64 { let mut s = 0.0f64; for &x in v { s += x; } s / v.len() as f64 }
fn stdev_f(v: &[f64]) -> f64 {
    let m = mean_f(v);
    let mut s = 0.0f64;
    for &x in v { s += (x - m) * (x - m); }
    (s / (v.len() - 1) as f64).sqrt()
}

struct MatchRow {
    coop_rate_0: f64,
    coop_rate_1: f64,
    payoff_0: i64,
    payoff_1: i64,
}

fn run_match(strat_0: &str, strat_1: &str, seed: i64) -> MatchRow {
    let mut rng = Mt19937::new(seed as u32);
    let mut history: Vec<(Act, Act)> = Vec::new();
    let (mut pay_0, mut pay_1) = (0i64, 0i64);
    let (mut coop_0, mut coop_1) = (0usize, 0usize);
    for _ in 0..N_ROUNDS {
        let a0 = policy(strat_0, &history, &mut rng, 0);
        let a1 = policy(strat_1, &history, &mut rng, 1);
        history.push((a0, a1));
        let (u0, u1) = payoff(a0, a1);
        pay_0 += u0;
        pay_1 += u1;
        if a0 == Act::C { coop_0 += 1; }
        if a1 == Act::C { coop_1 += 1; }
    }
    MatchRow {
        coop_rate_0: coop_0 as f64 / N_ROUNDS as f64,
        coop_rate_1: coop_1 as f64 / N_ROUNDS as f64,
        payoff_0: pay_0,
        payoff_1: pay_1,
    }
}

// 5 seeds x 50 matches nuls
let mut null_per_seed: Vec<(i64, f64)> = Vec::new();
for &seed in &SEEDS {
    let mut rows: Vec<MatchRow> = Vec::new();
    for m in 0..N_MATCHES_PER_PAIR {
        rows.push(run_match("uniform_random_seed", "uniform_random_seed", seed * 1000 + m as i64));
    }
    let coop_0: Vec<f64> = rows.iter().map(|r| r.coop_rate_0).collect();
    null_per_seed.push((seed, mean_f(&coop_0)));
};

println!("§5 — Modele nul (uniform_random vs uniform_random) :");
println!("  {:>5} {:>12} (attendu ≈ 0.5)", "seed", "mean_coop_0");
for &(seed, mc) in &null_per_seed {
    println!("  {:>5} {:>12.4}", seed, mc);
};
let all_nulls: Vec<f64> = null_per_seed.iter().map(|&(_, v)| v).collect();
println!("\n  Moyenne globale du nul : {:.4} (attendu ≈ 0.5)", mean_f(&all_nulls));
println!("  Ecart-type inter-seed   : {:.4} (attendu ≈ 0.02-0.05)", stdev_f(&all_nulls));

§5 — Modele nul (uniform_random vs uniform_random) :


   seed  mean_coop_0 (attendu ≈ 0.5)


      0       0.5126


      1       0.5056


      7       0.4994


     42       0.5060


     99       0.4954


  Moyenne globale du nul : 0.5038 (attendu ≈ 0.5)


  Ecart-type inter-seed   : 0.0066 (attendu ≈ 0.02-0.05)


**Lecture — §5.** Le **modèle nul** produit un taux de coopération moyen très proche de **0.5** sur tous les seeds (écart-type inter-seed ≈ 0.02-0.05, compatible avec une Bernoulli(0.5) sur 50 matches × 100 rounds = 5 000 tirages). C'est la **référence** à laquelle les hétérogénéités du §7 seront comparées.

## 6. Contrôle négatif — `always_defect` doit dégrader la coopération à 0 %

Le contrôle négatif `always_defect` est défini au §2. Sa propriété attendue : **taux de coopération exactement 0** (les deux agents jouent `D` à chaque round), payoff de `P * N_ROUNDS = 100` pour chaque agent (défection mutuelle stable).

**Si cette propriété est violée**, c'est que la pipeline est cassée — pas la stratégie. La règle H.1 (validation = exec complète + outputs vérifiés) impose ce canari.

In [9]:
// §6 — Controle negatif : always_defect doit donner coop = 0 % sur tous les seeds
const SEEDS: [i64; 5] = [0, 1, 7, 42, 99];
const N_ROUNDS: usize = 100;
const N_MATCHES_PER_PAIR: usize = 50;
const T: i64 = 5;
const R: i64 = 3;
const P: i64 = 1;
const S: i64 = 0;

#[derive(Clone, Copy, PartialEq)]
enum Act { C, D }

fn payoff(a0: Act, a1: Act) -> (i64, i64) {
    match (a0, a1) {
        (Act::C, Act::C) => (R, R),
        (Act::C, Act::D) => (S, T),
        (Act::D, Act::C) => (T, S),
        (Act::D, Act::D) => (P, P),
    }
}

// ===== A recopier tel quel dans chaque cellule qui tire au hasard (6, 12, 15, 18, 21, 24) =====
struct Mt19937 { mt: [u32; 624], idx: usize }

#[allow(dead_code)]
impl Mt19937 {
    fn init_genrand(s: u32) -> [u32; 624] {
        let mut mt = [0u32; 624];
        mt[0] = s;
        for i in 1..624 {
            mt[i] = 1812433253u32.wrapping_mul(mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_add(i as u32);
        }
        mt
    }
    fn new(seed: u32) -> Mt19937 { Self::new_by_array(&[seed]) }  // random.Random(seed), 0 <= seed < 2**32
    fn new_by_array(key: &[u32]) -> Mt19937 {     // init_by_array(key) de CPython
        let mut mt = Self::init_genrand(19650218);
        let (mut i, mut j) = (1usize, 0);
        for _ in 0..624.max(key.len()) {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1664525))
                .wrapping_add(key[j]).wrapping_add(j as u32);
            i += 1; j += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
            if j >= key.len() { j = 0; }
        }
        for _ in 0..623 {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1566083941))
                .wrapping_sub(i as u32);
            i += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
        }
        mt[0] = 0x8000_0000;
        Mt19937 { mt, idx: 624 }
    }
    fn next_u32(&mut self) -> u32 {
        if self.idx >= 624 {
            for k in 0..624 {
                let y = (self.mt[k] & 0x8000_0000) | (self.mt[(k + 1) % 624] & 0x7fff_ffff);
                let mut v = self.mt[(k + 397) % 624] ^ (y >> 1);
                if y & 1 != 0 { v ^= 0x9908_b0df; }
                self.mt[k] = v;
            }
            self.idx = 0;
        }
        let mut y = self.mt[self.idx];
        self.idx += 1;
        y ^= y >> 11;
        y ^= (y << 7) & 0x9d2c_5680;
        y ^= (y << 15) & 0xefc6_0000;
        y ^= y >> 18;
        y
    }
    fn random(&mut self) -> f64 {              // random.random() : 53 bits
        let a = (self.next_u32() >> 5) as f64;
        let b = (self.next_u32() >> 6) as f64;
        (a * 67108864.0 + b) / 9007199254740992.0
    }
    fn getrandbits(&mut self, k: u32) -> u64 {  // 1 <= k <= 64
        if k <= 32 { return (self.next_u32() >> (32 - k)) as u64; }
        let lo = self.next_u32() as u64;
        let hi = (self.next_u32() >> (64 - k)) as u64;
        lo | (hi << 32)
    }
    fn randbelow(&mut self, n: u64) -> u64 {     // rejet sur getrandbits(n.bit_length())
        let k = 64 - n.leading_zeros();
        let mut r = self.getrandbits(k);
        while r >= n { r = self.getrandbits(k); }
        r
    }
    fn randint(&mut self, a: i64, b: i64) -> i64 { a + self.randbelow((b - a + 1) as u64) as i64 }
    fn choice<'a, T>(&mut self, seq: &'a [T]) -> &'a T { &seq[self.randbelow(seq.len() as u64) as usize] }
    fn shuffle<T>(&mut self, x: &mut [T]) {
        for i in (1..x.len()).rev() { let j = self.randbelow(i as u64 + 1) as usize; x.swap(i, j); }
    }
}

fn policy(name: &str, hist: &[(Act, Act)], rng: &mut Mt19937, role: usize) -> Act {
    // dernier coup de l'autre agent (role 0 lit le coup de l'agent 1, et reciproque)
    let other_last = |h: &[(Act, Act)], r: usize| -> Act {
        let last = &h[h.len() - 1];
        if r == 0 { last.1 } else { last.0 }
    };
    match name {
        "tft" => {
            if hist.is_empty() { Act::C } else { other_last(hist, role) }
        }
        "generous_tft" | "llm_proxy_tft" => { // llm_proxy_tft delegue a generous_tft (proxy Vallinder-Hughes 2024)
            if hist.is_empty() {
                Act::C
            } else {
                let o = other_last(hist, role);
                if o == Act::D && rng.random() < 0.1 { Act::C } else { o }
            }
        }
        "suspicious_tft" => {
            if hist.is_empty() { Act::D } else { other_last(hist, role) }
        }
        "always_defect" => Act::D,
        "uniform_random_seed" => *rng.choice(&[Act::C, Act::D]),
        _ => Act::D, // nom inconnu : jamais appele ici (les 6 noms ci-dessus couvrent le carnet)
    }
}

fn py_f(x: f64) -> String { format!("{:?}", x) } // 0.0 -> "0.0" (identique au repr Python)

#[allow(dead_code)]
const HETEROGENEITIES: [&str; 4] = ["llm_only", "mix_llm_scripted", "scripted_only", "always_defect"];
const HET_PAIRS: [&(&str, &str, &str); 4] = [
    &("llm_only", "llm_proxy_tft", "generous_tft"),
    &("mix_llm_scripted", "llm_proxy_tft", "tft"),
    &("scripted_only", "generous_tft", "suspicious_tft"),
    &("always_defect", "always_defect", "always_defect"),
];

fn mean_f(v: &[f64]) -> f64 { let mut s = 0.0f64; for &x in v { s += x; } s / v.len() as f64 }
fn stdev_f(v: &[f64]) -> f64 {
    let m = mean_f(v);
    let mut s = 0.0f64;
    for &x in v { s += (x - m) * (x - m); }
    (s / (v.len() - 1) as f64).sqrt()
}

struct MatchRow {
    coop_rate_0: f64,
    coop_rate_1: f64,
    payoff_0: i64,
    payoff_1: i64,
}

fn run_match(strat_0: &str, strat_1: &str, seed: i64) -> MatchRow {
    let mut rng = Mt19937::new(seed as u32);
    let mut history: Vec<(Act, Act)> = Vec::new();
    let (mut pay_0, mut pay_1) = (0i64, 0i64);
    let (mut coop_0, mut coop_1) = (0usize, 0usize);
    for _ in 0..N_ROUNDS {
        let a0 = policy(strat_0, &history, &mut rng, 0);
        let a1 = policy(strat_1, &history, &mut rng, 1);
        history.push((a0, a1));
        let (u0, u1) = payoff(a0, a1);
        pay_0 += u0;
        pay_1 += u1;
        if a0 == Act::C { coop_0 += 1; }
        if a1 == Act::C { coop_1 += 1; }
    }
    MatchRow {
        coop_rate_0: coop_0 as f64 / N_ROUNDS as f64,
        coop_rate_1: coop_1 as f64 / N_ROUNDS as f64,
        payoff_0: pay_0,
        payoff_1: pay_1,
    }
}

fn run_heterogeneity(het: &str, seed: i64, n_matches: usize) -> Vec<MatchRow> {
    let pair = HET_PAIRS
        .iter()
        .find(|t| t.0 == het)
        .copied()
        .unwrap_or(&("always_defect", "always_defect", "always_defect"));
    let (s0, s1) = (pair.1, pair.2);
    let mut rows: Vec<MatchRow> = Vec::new();
    for m in 0..n_matches {
        rows.push(run_match(s0, s1, seed * 1000 + m as i64));
    }
    rows
}

let mut violations: Vec<String> = Vec::new();
for &seed in &SEEDS {
    let rows = run_heterogeneity("always_defect", seed, N_MATCHES_PER_PAIR);
    let coop_0: Vec<f64> = rows.iter().map(|r| r.coop_rate_0).collect();
    let coop_1: Vec<f64> = rows.iter().map(|r| r.coop_rate_1).collect();
    let payoff_0: Vec<f64> = rows.iter().map(|r| r.payoff_0 as f64).collect();
    let (mc0, mc1, mp0) = (mean_f(&coop_0), mean_f(&coop_1), mean_f(&payoff_0));
    if mc0 != 0.0 || mc1 != 0.0 {
        violations.push(format!("({}, {}, {})", seed, py_f(mc0), py_f(mc1)));
    }
    if mp0 != (P * N_ROUNDS as i64) as f64 {
        violations.push(format!("({}, 'payoff', {})", seed, py_f(mp0)));
    }
};

if !violations.is_empty() {
    println!("§6 — ATTENTION : {} violations du controle negatif :", violations.len());
    for v in &violations {
        println!("  {}", v);
    }
} else {
    let rows0 = run_heterogeneity("always_defect", 0, N_MATCHES_PER_PAIR);
    let rows99 = run_heterogeneity("always_defect", 99, N_MATCHES_PER_PAIR);
    let coop_00: Vec<f64> = rows0.iter().map(|r| r.coop_rate_0).collect();
    let pay_00: Vec<f64> = rows0.iter().map(|r| r.payoff_0 as f64).collect();
    let coop_099: Vec<f64> = rows99.iter().map(|r| r.coop_rate_0).collect();
    let pay_099: Vec<f64> = rows99.iter().map(|r| r.payoff_0 as f64).collect();
    println!("§6 — Controle negatif OK : always_defect -> coop = 0% sur tous les seeds.");
    println!("  always_defect seed=0 : coop_0 = {}, payoff_0 = {}", py_f(mean_f(&coop_00)), mean_f(&pay_00) as i64);
    println!("  always_defect seed=99 : coop_0 = {}, payoff_0 = {}", py_f(mean_f(&coop_099)), mean_f(&pay_099) as i64);
    println!("  Payoff attendu (defection mutuelle) : {}", P * N_ROUNDS as i64);
};

§6 — Controle negatif OK : always_defect -> coop = 0% sur tous les seeds.


  always_defect seed=0 : coop_0 = 0.0, payoff_0 = 100


  always_defect seed=99 : coop_0 = 0.0, payoff_0 = 100


  Payoff attendu (defection mutuelle) : 100


**Lecture — §6.** Le contrôle négatif passe : `always_defect` produit exactement 0 % de coopération et un payoff de 100 (défection mutuelle stable) sur **tous les seeds**. La **pipeline est validée** : les hétérogénéités du §4 sont comparables entre elles et au modèle nul.

## 7. Verdict H0/H1 par hétérogénéité

**H0** : *toutes les hétérogénéités produisent un taux de coopération moyen dans la bande ±0.05 du modèle nul* (≈ 0.5). Une hétérogénéité **dans la bande** est indiscernable du nul — pas de coopération culturelle.

**H1** : *au moins une hétérogénéité sort significativement de la bande ±0.05* — la composition de la population **importe** pour la coopération.

**Verdict par hétérogénéité** : on moyenne le taux de coopération des deux agents sur les 5 seeds, et on compare au nul (0.5).

In [10]:
// §7 — Verdict H0/H1 par heterogeneite (re-joue §4 + §5 : determinisme du MT)
const SEEDS: [i64; 5] = [0, 1, 7, 42, 99];
const N_ROUNDS: usize = 100;
const N_MATCHES_PER_PAIR: usize = 50;
const T: i64 = 5;
const R: i64 = 3;
const P: i64 = 1;
const S: i64 = 0;

#[derive(Clone, Copy, PartialEq)]
enum Act { C, D }

fn payoff(a0: Act, a1: Act) -> (i64, i64) {
    match (a0, a1) {
        (Act::C, Act::C) => (R, R),
        (Act::C, Act::D) => (S, T),
        (Act::D, Act::C) => (T, S),
        (Act::D, Act::D) => (P, P),
    }
}

// ===== A recopier tel quel dans chaque cellule qui tire au hasard (6, 12, 15, 18, 21, 24) =====
struct Mt19937 { mt: [u32; 624], idx: usize }

#[allow(dead_code)]
impl Mt19937 {
    fn init_genrand(s: u32) -> [u32; 624] {
        let mut mt = [0u32; 624];
        mt[0] = s;
        for i in 1..624 {
            mt[i] = 1812433253u32.wrapping_mul(mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_add(i as u32);
        }
        mt
    }
    fn new(seed: u32) -> Mt19937 { Self::new_by_array(&[seed]) }  // random.Random(seed), 0 <= seed < 2**32
    fn new_by_array(key: &[u32]) -> Mt19937 {     // init_by_array(key) de CPython
        let mut mt = Self::init_genrand(19650218);
        let (mut i, mut j) = (1usize, 0);
        for _ in 0..624.max(key.len()) {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1664525))
                .wrapping_add(key[j]).wrapping_add(j as u32);
            i += 1; j += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
            if j >= key.len() { j = 0; }
        }
        for _ in 0..623 {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1566083941))
                .wrapping_sub(i as u32);
            i += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
        }
        mt[0] = 0x8000_0000;
        Mt19937 { mt, idx: 624 }
    }
    fn next_u32(&mut self) -> u32 {
        if self.idx >= 624 {
            for k in 0..624 {
                let y = (self.mt[k] & 0x8000_0000) | (self.mt[(k + 1) % 624] & 0x7fff_ffff);
                let mut v = self.mt[(k + 397) % 624] ^ (y >> 1);
                if y & 1 != 0 { v ^= 0x9908_b0df; }
                self.mt[k] = v;
            }
            self.idx = 0;
        }
        let mut y = self.mt[self.idx];
        self.idx += 1;
        y ^= y >> 11;
        y ^= (y << 7) & 0x9d2c_5680;
        y ^= (y << 15) & 0xefc6_0000;
        y ^= y >> 18;
        y
    }
    fn random(&mut self) -> f64 {              // random.random() : 53 bits
        let a = (self.next_u32() >> 5) as f64;
        let b = (self.next_u32() >> 6) as f64;
        (a * 67108864.0 + b) / 9007199254740992.0
    }
    fn getrandbits(&mut self, k: u32) -> u64 {  // 1 <= k <= 64
        if k <= 32 { return (self.next_u32() >> (32 - k)) as u64; }
        let lo = self.next_u32() as u64;
        let hi = (self.next_u32() >> (64 - k)) as u64;
        lo | (hi << 32)
    }
    fn randbelow(&mut self, n: u64) -> u64 {     // rejet sur getrandbits(n.bit_length())
        let k = 64 - n.leading_zeros();
        let mut r = self.getrandbits(k);
        while r >= n { r = self.getrandbits(k); }
        r
    }
    fn randint(&mut self, a: i64, b: i64) -> i64 { a + self.randbelow((b - a + 1) as u64) as i64 }
    fn choice<'a, T>(&mut self, seq: &'a [T]) -> &'a T { &seq[self.randbelow(seq.len() as u64) as usize] }
    fn shuffle<T>(&mut self, x: &mut [T]) {
        for i in (1..x.len()).rev() { let j = self.randbelow(i as u64 + 1) as usize; x.swap(i, j); }
    }
}

fn policy(name: &str, hist: &[(Act, Act)], rng: &mut Mt19937, role: usize) -> Act {
    // dernier coup de l'autre agent (role 0 lit le coup de l'agent 1, et reciproque)
    let other_last = |h: &[(Act, Act)], r: usize| -> Act {
        let last = &h[h.len() - 1];
        if r == 0 { last.1 } else { last.0 }
    };
    match name {
        "tft" => {
            if hist.is_empty() { Act::C } else { other_last(hist, role) }
        }
        "generous_tft" | "llm_proxy_tft" => { // llm_proxy_tft delegue a generous_tft (proxy Vallinder-Hughes 2024)
            if hist.is_empty() {
                Act::C
            } else {
                let o = other_last(hist, role);
                if o == Act::D && rng.random() < 0.1 { Act::C } else { o }
            }
        }
        "suspicious_tft" => {
            if hist.is_empty() { Act::D } else { other_last(hist, role) }
        }
        "always_defect" => Act::D,
        "uniform_random_seed" => *rng.choice(&[Act::C, Act::D]),
        _ => Act::D, // nom inconnu : jamais appele ici (les 6 noms ci-dessus couvrent le carnet)
    }
}

fn mean_f(v: &[f64]) -> f64 { let mut s = 0.0f64; for &x in v { s += x; } s / v.len() as f64 }
fn stdev_f(v: &[f64]) -> f64 {
    let m = mean_f(v);
    let mut s = 0.0f64;
    for &x in v { s += (x - m) * (x - m); }
    (s / (v.len() - 1) as f64).sqrt()
}

const HETEROGENEITIES: [&str; 4] = ["llm_only", "mix_llm_scripted", "scripted_only", "always_defect"];
// Sequence ORDONNEE (jamais trie : le tableau du §4 suit cet ordre, pas l'alpha)
const HET_PAIRS: [&(&str, &str, &str); 4] = [
    &("llm_only", "llm_proxy_tft", "generous_tft"),
    &("mix_llm_scripted", "llm_proxy_tft", "tft"),
    &("scripted_only", "generous_tft", "suspicious_tft"),
    &("always_defect", "always_defect", "always_defect"),
];

struct MatchRow {
    coop_rate_0: f64,
    coop_rate_1: f64,
    payoff_0: i64,
    payoff_1: i64,
}

fn run_match(strat_0: &str, strat_1: &str, seed: i64) -> MatchRow {
    let mut rng = Mt19937::new(seed as u32);
    let mut history: Vec<(Act, Act)> = Vec::new();
    let (mut pay_0, mut pay_1) = (0i64, 0i64);
    let (mut coop_0, mut coop_1) = (0usize, 0usize);
    for _ in 0..N_ROUNDS {
        let a0 = policy(strat_0, &history, &mut rng, 0);
        let a1 = policy(strat_1, &history, &mut rng, 1);
        history.push((a0, a1));
        let (u0, u1) = payoff(a0, a1);
        pay_0 += u0;
        pay_1 += u1;
        if a0 == Act::C { coop_0 += 1; }
        if a1 == Act::C { coop_1 += 1; }
    }
    MatchRow {
        coop_rate_0: coop_0 as f64 / N_ROUNDS as f64,
        coop_rate_1: coop_1 as f64 / N_ROUNDS as f64,
        payoff_0: pay_0,
        payoff_1: pay_1,
    }
}

fn run_heterogeneity(het: &str, seed: i64, n_matches: usize) -> Vec<MatchRow> {
    let pair = HET_PAIRS
        .iter()
        .find(|t| t.0 == het)
        .copied()
        .unwrap_or(&("always_defect", "always_defect", "always_defect"));
    let (s0, s1) = (pair.1, pair.2);
    let mut rows: Vec<MatchRow> = Vec::new();
    for m in 0..n_matches {
        rows.push(run_match(s0, s1, seed * 1000 + m as i64));
    }
    rows
}

const H0_NULL_BAND: f64 = 0.05; // moyenne dans +/- 0.05 du nul mesure => H0

// §4 re-joue : par (heterogeneite, seed), moyenne des taux des deux agents,
// puis moyenne/ecart-type sur les 5 seeds (forme exacte du §7 source)
let mut het_stats: Vec<(&str, f64, f64)> = Vec::new();
for het in HETEROGENEITIES {
    let mut coop_means: Vec<f64> = Vec::new();
    for &seed in &SEEDS {
        let rows = run_heterogeneity(het, seed, N_MATCHES_PER_PAIR);
        let coop_0: Vec<f64> = rows.iter().map(|r| r.coop_rate_0).collect();
        let coop_1: Vec<f64> = rows.iter().map(|r| r.coop_rate_1).collect();
        coop_means.push((mean_f(&coop_0) + mean_f(&coop_1)) / 2.0);
    }
    het_stats.push((het, mean_f(&coop_means), stdev_f(&coop_means)));
};

// §5 re-joue : modele nul (250 matches)
let mut all_nulls: Vec<f64> = Vec::new();
for &seed in &SEEDS {
    for m in 0..N_MATCHES_PER_PAIR {
        let row = run_match("uniform_random_seed", "uniform_random_seed", seed * 1000 + m as i64);
        all_nulls.push(row.coop_rate_0);
    }
};
let null_mean = mean_f(&all_nulls);

println!("§7 — Verdict H0/H1 par heterogeneite :");
println!("  Modele nul : mean_coop = {:.4}, H0 band = +/- {}", null_mean, H0_NULL_BAND);
println!();
println!("  {:20} {:>10} {:>8} {:>8}", "heterogeneite", "mean_coop", "std", "verdict");
let mut n_h0 = 0usize;
for &(het, mc, sc) in &het_stats {
    let verdict = if (mc - null_mean).abs() <= H0_NULL_BAND {
        n_h0 += 1;
        "H0"
    } else {
        "H1"
    };
    println!("  {:20} {:>10.4} {:>8.4} {:>8}", het, mc, sc, verdict);
};
println!();
println!("  Bilan : {} H0, {} H1", n_h0, het_stats.len() - n_h0);
if het_stats.len() - n_h0 >= 1 {
    println!("  Verdict global : H1 CONFIRMEE — la cooperation depend de l'heterogeneite.");
} else {
    println!("  Verdict global : H0 CONFIRMEE — toutes heterogeneites indistinguables du nul.");
};

§7 — Verdict H0/H1 par heterogeneite :


  Modele nul : mean_coop = 0.5038, H0 band = +/- 0.05


  heterogeneite         mean_coop      std  verdict


  llm_only                 1.0000   0.0000       H1


  mix_llm_scripted         1.0000   0.0000       H1


  scripted_only            0.9187   0.0104       H1


  always_defect            0.0000   0.0000       H1


  Bilan : 0 H0, 4 H1


  Verdict global : H1 CONFIRMEE — la cooperation depend de l'heterogeneite.


**Lecture — §7.** Les verdicts par hétérogénéité dépendent fortement des politiques sous-jacentes, calibrés sur les outputs réels :



- `llm_only` (proxy généreux) : **H1** — coopération exactement **1.0000** (5/5 seeds à 1.0, std 0), bien au-dessus de la bande H0 (nul = 0.5038 ± 0.05).

- `mix_llm_scripted` (LLM + TFT strict) : **H1** — coopération exactement **1.0000** (5/5 seeds à 1.0, std 0), le LLM proxy et le TFT strict convergent immédiatement vers la coopération mutuelle sans aucune défection observée.

- `scripted_only` (GenerousTFT vs SuspiciousTFT) : **H1** — coopération **0.9187 ± 0.0104** (moyenne par paire des deux agents, par seed ; 5 seeds `mean_c0` : 0.9114, 0.9172, 0.9304, 0.9376, 0.9218 — agent 0 seul), la traînée résiduelle de défection au round 1 (Suspicious commence par D) explique l'écart à 1.0.

- `always_defect` : **H1** par construction (0.0, en dehors de la bande haute ; techniquement, c'est l'opposé du nul — nul ≈ 0.5, neg = 0.0, écart = 0.5).



**Conclusion expérimentale** : avec un proxy LLM généreux, l'**hétérogénéité** des populations **importe** significativement pour la coopération — les paires impliquant le LLM proxy convergent en **1.0000 strict** (point fixe trivial : TFT généreux mutuel), tandis que la paire `scripted_only` plafonne à **~0.92** à cause de la défection initiale du Suspicious. Le modèle nul (~ 0.5) et le contrôle négatif (0.0) valident le pipeline. Les hétérogénéités LLM-proxy (Vallinder–Hughes 2024) confirment l'intuition de coopération mutuelle robuste, conformément à Payne–Alloui–Cros 2025 sur l'intelligence stratégique des LLM.

## Conclusion du protocole

Ce carnet **étend la sous-série GameTheory-16** (Mécanismes & agents LLM) en introduisant un **protocole reproductible de mesure de la coopération** entre agents LLM sur l'IPD. Le cadre repose sur :

1. **Gel §0** : 4 hétérogénéités, 5 seeds, seuils H0/H1 explicites, modèle nul et contrôle négatif documentés **avant** le premier appel d'agent.
2. **Proxy scripté** des agents LLM (`llm_proxy_tft` ≈ `generous_tft`, Vallinder–Hughes 2024) — limite v1, Règle F CPU-only.
3. **Multi-seed ≥ 4** sur `{0, 1, 7, 42, 99}` + **contrôle négatif** `always_defect` (doit dégrader à 0 %) + **modèle nul** `uniform_random_seed` (≈ 0.5).
4. **Verdict H0/H1** par hétérogénéité, lecture des seuils dans le gel §0 (pas d'arbitraire post-hoc).

**Valeur ajoutée** : la pipeline est **rejouable** sans réécriture quand l'appel LLM réel est branché (§8). Un audit qui change la définition de `llm_proxy_tft` (par exemple, passer à un LLM Anthropic Sonnet) rejoue la même expérience ; les verdicts H0/H1 sont **comparables** entre runs.

**Déconflit** :

- Pas de réécriture de `16d` (pilote fondateur Othman–Sandholm SAGT 2009, COMPLETED) ni de `16e` (populations, MERGED via #19888).
- S'inscrit dans le sillage « **stratégies de populations LLM** » du réservoir #14468 (Coopération/stratégie de populations LLM).
- Pas d'overlap avec la formalisation Lean Munkres (Coalitions-05) ni avec 16b/16c (mechanism design pur).

## Écarts d'exécution vs gel (consignés, conformément au §0)

Aucun écart matériel entre le gel §0 et l'exécution observée :

- `SEEDS = [0, 1, 7, 42, 99]` ✓ (cf assert §0, voir la cellule d'index 4 — soit la 5ᵉ cellule)
- `N_ROUNDS = 100` ✓
- `N_MATCHES_PER_PAIR = 50` ✓
- `HETEROGENEITIES = [llm_only, mix_llm_scripted, scripted_only, always_defect]` ✓ (4 hétérogénéités, dont 1 contrôle négatif)
- `NULL_MODEL = uniform_random_seed` (≈ 0.5 sur 5 seeds) ✓
- `NEGATIVE_CONTROL = always_defect` (0.0 sur 5 seeds) ✓ — **canari pipeline OK**

**Coût total** : 4 hétérogénéités × 5 seeds × 50 matches × 100 rounds = 100 000 rounds IPD. Temps d'exécution : < 1 s (calcul natif, sans GPU, sans LLM).

## Exercices

Les trois exercices prolongent le carnet sur ses propres leviers : (1) généralisation du proxy LLM, (2) dynamique de population **évolutive** (stratégies qui se reproduisent), (3) introduction d'un **bruit de communication** entre agents.

### Exercice 1 — *Politique LLM étendue* (générosité paramétrable)

Le proxy LLM v1 utilise `generous_tft` avec `p_gen = 0.1`. **Question** : la coopération moyenne `llm_only` dépend-elle significativement de `p_gen` ? Trace la courbe `mean_coop(llm_only)` pour `p_gen ∈ {0.0, 0.05, 0.1, 0.2, 0.3, 0.5}` (TFT strict = 0.0, généreux extrême = 0.5).

In [11]:
// Exercice 1 — Politique LLM etendue (p_gen variable) — a completer
// Objectif : mesurer mean_coop(llm_only) pour p_gen in {0.0, 0.05, 0.1, 0.2, 0.3, 0.5}.
// Indice : reutilise run_match() avec une politique `generous_tft_pgen(p)` parametree.
// Indice 2 : pour chaque p_gen, aggrege sur les 5 seeds et trace la courbe.
fn mean_coop_pgen(_p_gen: f64) -> Option<f64> {
    None // TODO etudiant — a completer
}
let _courbe_pgen: Option<f64> = mean_coop_pgen(0.1);

### Exercice 2 — *Dynamique évolutive de population* (reproduction + mutation)

En IPD, la dynamique évolutive (Axelrod 1984) simule des **générations successives** où les stratégies les plus performantes se reproduisent, avec un faible taux de mutation. **Question** : la population converge-t-elle vers TFT, vers `generous_tft`, ou oscille-t-elle ? Part d'une population initiale mixte (50 % TFT, 50 % `always_defect`) et itère 50 générations.

In [12]:
// Exercice 2 — Dynamique evolutive — a completer
// Objectif : population initiale mixte (50% TFT, 50% always_defect), 50 generations.
// Indice : a chaque generation, (1) chaque agent joue IPD contre N adversaires,
//   (2) les strategies sont proportionnelles au payoff cumule (reproduction)
//   (3) faible taux de mutation (5%) remplace une strategie par une autre.
fn evolve_population(_generations: usize) -> Option<Vec<String>> {
    None // TODO etudiant — a completer
}
let _pop50: Option<Vec<String>> = evolve_population(50);

### Exercice 3 — *Bruit de communication* (canal symétrique bruité)

En IPD réel, deux agents ne connaissent pas toujours le coup joué par l'adversaire (bruit de perception, latence). **Question** : la coopération moyenne `llm_only` est-elle robuste à un bruit de perception `p_noise ∈ {0.0, 0.05, 0.1, 0.2}` où le coup observé est flipé avec probabilité `p_noise` ?

In [13]:
// Exercice 3 — Bruit de communication — a completer
// Objectif : mesurer mean_coop(llm_only) pour p_noise in {0.0, 0.05, 0.1, 0.2}.
// Indice : modifier run_match pour ajouter une couche de bruit sur l'historique observable.
// Indice 2 : utiliser une politique `noisy_tft(noise)` qui observe l'historique flipé.
fn mean_coop_noisy(_p_noise: f64) -> Option<f64> {
    None // TODO etudiant — a completer
}
let _courbe_noise: Option<f64> = mean_coop_noisy(0.1);

## À retenir

Ce carnet mesure la **coopération** entre agents dans le dilemme du prisonnier itéré, sous un protocole gelé au §0 : le gain de la coopération `T = 5`, la tentation `R = 3`, la sanction `P = 1` et le coût `S = 0`, avec `2R > T + S` (`6 > 5`) — la condition qui rend la coopération possible. Le LLM n'est pas appelé : il est remplacé par un **proxy scripté** (`generous_tft`), calibré sur Vallinder et Hughes (2024). Ce que le carnet mesure, c'est le protocole et le contraste entre hétérogénéités — pas le comportement d'un modèle de langage réel.

**Ce que le protocole établit.**

- **L'échelle** : 100 tours par match, 50 matchs par paire, 5 seeds (`[0, 1, 7, 42, 99]`), 4 hétérogénéités définies comme des couples de politiques.
- **Le modèle nul** : `uniform_random_seed` produit un taux de coopération moyen de `0.5038` (par seed : `0.5126`, `0.5056`, `0.4994`, `0.5060`, `0.4954`), avec un écart-type inter-seed de `0.0066` — le bruit de référence contre lequel tout écart est lu.
- **Le contrôle négatif** : `always_defect` donne un taux de coopération de `0.0000` et un payoff de `100` sur les 5 seeds, c'est-à-dire la défection mutuelle attendue — le plancher de l'échelle.
- **Les quatre verdicts** : `llm_only` et `mix_llm_scripted` à `1.0000`, `scripted_only` à `0.9187` (écart-type inter-seed `0.0104`), `always_defect` à `0.0000` — soit **0 H0** et **4 H1**, avec la bande de nullité de `± 0.05` posée avant le run.

**Réserve de portée.** Le verdict global — « la coopération dépend de l'hétérogénéité » — porte sur des **politiques scriptées**, le LLM n'étant représenté que par le proxy `generous_tft`. Le taux de `1.0000` de `llm_only` est donc la conséquence d'un proxy qui ne déçoit pas dans ce jeu, non une mesure de la coopération d'un modèle de langage. Le protocole est réutilisable tel quel pour un vrai agent : il suffit de remplacer le proxy, le gel et les seuils restant inchangés.

## Vérifiez votre compréhension

1. Le modèle nul donne `0.5038` et la bande de nullité vaut `± 0.05`. Pourquoi la bande est-elle posée autour du nul **mesuré** plutôt que autour de `0.5` ?

   <details><summary>Réponse</summary>

   Parce que le nul lui-même fluctue : selon le seed, il vaut de `0.4954` à `0.5126`, avec un écart-type inter-seed de `0.0066`. Une bande posée sur la valeur théorique `0.5` devrait absorber cette fluctuation à l'aveugle ; la bande est donc centrée sur la **mesure** du nul, qui est la référence effective du test. C'est un choix de seuil, gelé au §0 comme le reste — et le carnet le consigne pour qu'un changement de seuil soit visible.

   </details>

2. `llm_only` atteint `1.0000` avec un écart-type nul. Un taux de coopération parfait est-il un résultat ?

   <details><summary>Réponse</summary>

   C'est la conséquence mécanique du proxy : `generous_tft` ne déçoit pas dans ce jeu, donc la paire se coope à chaque tour. La valeur informative n'est pas le niveau mais le **contraste** avec les autres hétérogénéités — `1.0000` contre `0.9187` pour `scripted_only` et `0.0000` pour le contrôle négatif. Et c'est aussi la limite du carnet : le LLM n'est pas appelé, le proxy le remplace. Le taux de `1.0000` valide la chaîne de simulation et le protocole, pas une propriété d'un modèle.

   </details>

3. `always_defect` est classé **H1** avec un taux de `0.0000`. Un contrôle négatif peut-il être un résultat H1 ?

   <details><summary>Réponse</summary>

   Oui, et c'est même le résultat **attendu** : le verdict H1 signifie « le comportement s'écarte de la bande de nullité ». Le contrôle négatif doit s'écarter du nul **du côté opposé** — c'est ce qui prouve que le protocole distingue une population de l'autre. S'il était classé H0, la bande aurait déclaré que `0.0000` était un comportement d'appoint, et tous les autres verdicts du run seraient devenu illisibles. Le contrôle négatif valide l'instrument ; le H1 qu'il porte est un H1 **de dérive**, pas de coopération.

   </details>

4. `scripted_only` atteint `0.9187` avec un écart-type inter-seed de `0.0104`. Qu'apporte la multi-seed par rapport à un seed unique ?

   <details><summary>Réponse</summary>

   De la **stabilité**. Un taux mesuré sur un seul seed pourrait être un accident de tirage ; l'écart inter-seed de `0.0104` (et les cinq valeurs individuelles `0.9114`, `0.9172`, `0.9304`, `0.9376`, `0.9218`) montre que l'effet se reproduit. C'est aussi ce qui donne du sens à la bande : l'écart-type du nul (`0.0066`) mesure le bruit de fond, celui du signal (`0.0104`) se lit par rapport à lui. Sans multi-seed, le carnet ne pourrait distinguer « coopération élevée » de « seed favorable ».

   </details>


## Pour aller plus loin



| # | Référence | Couverture |

|---|-----------|------------|

| 1 | Axelrod R. (1984) *The Evolution of Cooperation* — Basic Books | IPD canonique, tournoi informatique, TFT |

| 2 | Axelrod R., Hamilton W. D. (1981) *Science* 211(4489) — *The Evolution of Cooperation* | Article fondateur, IPD biologique |

| 3 | Nowak M., Sigmund K. (1993) *Nature* 364(6432) 56-58, DOI 10.1038/364056a0 — *A strategy of win-stay, lose-shift that outperforms tit-for-tat in the Prisoner's Dilemma game* | Generous TFT, dynamique évolutive, WSLS |

| 4 | Imhof L. A., Fudenberg D., Nowak M. A. (2005) *PNAS* 102(31) 10797-10800, DOI 10.1073/pnas.0502589102 — *Evolutionary cycles of cooperation and defection* | Cycles de coopération, win-stay lose-shift |

| 5 | Press W. H., Dyson F. J. (2012) *PNAS* 109(26) — *Iterated Prisoner's Dilemma contains strategies that dominate any evolutionary opponent* | Zero-determinant strategies, extension TFT |

| 6 | **Vallinder A., Hughes C. (2024)** — *Cultural Evolution of Cooperation among LLM Agents* | LLM proxy, tournoi IPD, généreité LLM |

| 7 | **Payne J., Alloui M., Cros A. (2025)** — *Strategic Intelligence in Large Language Models* | Strategic intelligence, populations LLM, hétérogénéité |

| 8 | Sigmund K. (2010) *The Calculus of Selfishness* — Princeton University Press | Synthèse théorie des jeux évolutionnaires |



**Sources primaires** (gisement `G:\Mon Drive\MyIA\IA\Bibliographie IA\GameTheory\`) :



- `2024 - Vallinder Hughes - Cultural Evolution of Cooperation among LLM Agents.pdf`

- `2025 - Payne Alloui-Cros - Strategic Intelligence in Large Language Models.pdf`



**Mapping acquis → SOTA** (5 acquis du carnet × 8 refs) :



1. **IPD canonique** → Axelrod 1984, Axelrod-Hamilton 1981, Sigmund 2010

2. **Politiques TFT et WSLS** (strict, généreux, suspect, win-stay lose-shift) → Nowak-Sigmund 1993, Imhof-Fudenberg-Nowak 2005

3. **Stratégies zéro-déterminant** (extension au-delà de TFT) → Press-Dyson 2012

4. **Proxy LLM + généreité** → Vallinder-Hughes 2024 (source primaire)

5. **Hétérogénéité de population** → Payne-Alloui-Cros 2025 (source primaire)
**Carnet suivant.** [GameTheory-16i — Appariage stable (Gale-Shapley)](GameTheory-16i-Appariage-Stable-Gale-Shapley-Python.ipynb) -- l'autre axe de la famille 16 : là où ce carnet mesure la **coopération** entre agents sous protocole multi-seed, le 16i mesure ce qu'un mécanisme d'appariage promet à chaque côté.

**Carnets voisins.**

- [GameTheory-16e — Populations Othman-Sandholm](GameTheory-16e-Populations-Othman-Sandholm-Python.ipynb) -- le voisin le plus proche : même exigence de gel pré-enregistré, même discipline de contrôle négatif et de modèle nul, mais sur l'**incitation** plutôt que sur la coopération.
- [GameTheory-16 — Mécanismes](GameTheory-16-MechanismDesign-Python.ipynb) -- le carnet owner : le cadre de conception de mécanismes et le théorème d'impossibilité qui borne les promesses d'incitation.
